# 🧠 Disease Prediction from Symptoms (Training Notebook)
This notebook prepares the dataset, trains a model using Keras, and exports it to TensorFlow Lite for offline mobile use.

In [1]:
# 📦 Step 1: Import Libraries
import pandas as pd
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from tensorflow import keras
from tensorflow.keras import layers
import pickle

In [2]:
import pandas as pd

# Load the dataset
df = pd.read_csv("../../../ai-microclinic-offline-dataset/Disease Symptom Prediction/dataset.csv")

# Identify all symptom columns (they start with 'Symptom_')
symptom_cols = [col for col in df.columns if col.startswith('Symptom')]

# Combine all symptom columns into a single list per row
df['Symptom List'] = df[symptom_cols].values.tolist()

# Remove None or NaN values from the list
df['Symptom List'] = df['Symptom List'].apply(lambda x: [s for s in x if pd.notna(s)])

# Keep only what's needed
df = df[['Disease', 'Symptom List']]
df.head()

,Disease,Symptom List
0,Fungal infection,"[itching, skin_rash, nodal_skin_eruptions, ..."
1,Fungal infection,"[ skin_rash, nodal_skin_eruptions, dischromi..."
2,Fungal infection,"[itching, nodal_skin_eruptions, dischromic _..."
3,Fungal infection,"[itching, skin_rash, dischromic _patches]"
4,Fungal infection,"[itching, skin_rash, nodal_skin_eruptions]"


In [3]:
# 🔢 Step 3: Encode Symptoms and Diseases
mlb = MultiLabelBinarizer()
X = mlb.fit_transform(df['Symptom List'])

diseases = df['Disease'].astype('category')
y = diseases.cat.codes
disease_mapping = dict(enumerate(diseases.cat.categories))
len(disease_mapping)

41

In [4]:
# 🧪 Step 4: Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [5]:
# 🤖 Step 5: Build and Train Model
model = keras.Sequential([
    layers.Input(shape=(X.shape[1],)),
    layers.Dense(64, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(len(disease_mapping), activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(X_train, y_train, epochs=20, batch_size=16, validation_split=0.1)

Epoch 1/20
222/222 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.5867 - loss: 2.7111 - val_accuracy: 1.0000 - val_loss: 0.1026
Epoch 2/20
222/222 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 1.0000 - loss: 0.0657 - val_accuracy: 1.0000 - val_loss: 0.0155
Epoch 3/20
222/222 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 1.0000 - loss: 0.0135 - val_accuracy: 1.0000 - val_loss: 0.0065
Epoch 4/20
222/222 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 1.0000 - loss: 0.0061 - val_accuracy: 1.0000 - val_loss: 0.0036
Epoch 5/20
222/222 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 1.0000 - loss: 0.0035 - val_accuracy: 1.0000 - val_loss: 0.0022
Epoch 6/20
222/222 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 1.0000 - loss: 0.0023 - val_accuracy: 1.0000 - val_loss: 0.0015
Epoch 7/20
222/222 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 1.0000 - loss: 0.0016 - val_accuracy: 1.0000 - val_loss: 0.0010
Epoch 8/20
222/222 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 1.0000 - loss: 0.0011 - val_accuracy: 1.

In [6]:
# 💾 Step 6: Save the Model
model.save("disease_model.h5")

In [7]:
# 💾 Step 7: Save Mappings
with open('../../android/app/src/main/assets/symptom_mapping.json', 'w') as f:
    import json
    json.dump(mlb.classes_.tolist(), f)

with open('../../android/app/src/main/assets/disease_mapping.json', 'w') as f:
    json.dump(disease_mapping, f)

In [8]:
# 🔁 Step 8: Convert to TFLite
import tensorflow as tf

model = tf.keras.models.load_model('disease_model.h5')
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open('../../android/app/src/main/assets/disease_model.tflite', 'wb') as f:
    f.write(tflite_model)

INFO:tensorflow:Assets written to: C:\Users\karti\AppData\Local\Temp\tmpln55ummu\assets


INFO:tensorflow:Assets written to: C:\Users\karti\AppData\Local\Temp\tmpln55ummu\assets


Saved artifact at 'C:\Users\karti\AppData\Local\Temp\tmpln55ummu'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 131), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 41), dtype=tf.float32, name=None)
Captures:
  3163821490000: TensorSpec(shape=(), dtype=tf.resource, name=None)
  3163821487888: TensorSpec(shape=(), dtype=tf.resource, name=None)
  3163821489808: TensorSpec(shape=(), dtype=tf.resource, name=None)
  3163821489040: TensorSpec(shape=(), dtype=tf.resource, name=None)
  3163820360912: TensorSpec(shape=(), dtype=tf.resource, name=None)
  3163820359760: TensorSpec(shape=(), dtype=tf.resource, name=None)
